In [3]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_columns', None)

In [4]:
phenos = pd.read_csv("/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno_deaggregated.csv")

In [5]:
phenos

,Unnamed: 0,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range
0,0,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,No,Yellow,No,Green,Dense,Dense,Brown,No,Yes,Mid Summer,Medium,Fall,Fall,No,abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN
1,1,Abies fraseri - fraser fir,abies fraseri,fraser fir,Abies,fraseri,ABFR,No,Purple,No,Dark Green,Moderate,Moderate,Brown,Yes,Yes,Mid Spring,Medium,Spring,Fall,No,abies fraseri,Non-flowering,N/a,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN
2,2,Acer ginnala - amur maple,acer ginnala,amur maple,Acer,ginnala,ACGI,Yes,White,No,Green,Dense,Moderate,Brown,Yes,No,Mid Spring,High,Summer,Fall,No,acer ginnala,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-03-01', '2026-03-02', '20..."
3,3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,Green,Dense,Porous,Brown,Yes,No,Early Spring,High,Summer,Fall,Yes,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-03-01', '2026-03-02', '20..."
4,4,Acer nigrum - black maple,acer nigrum,black maple,Acer,nigrum,ACNI5,Yes,Yellow,No,Green,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,Yes,acer nigrum,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-04-01', '2026-04-02', '20..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN
290,290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN
291,291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN
292,292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN


In [6]:
season_start = {"Spring": pd.to_datetime("2026-03-01"),
                "Summer": pd.to_datetime("2026-06-01"),
                "Fall": pd.to_datetime("2026-09-01"),
                "Winter": pd.to_datetime("2026-12-01")}

season_end = {"Spring": pd.to_datetime("2026-05-31 23:59:59.999999"),
                "Summer": pd.to_datetime("2026-08-31 23:59:59.999999"),
                "Fall": pd.to_datetime("2026-11-30 23:59:59.999999"),
                "Winter": pd.to_datetime("2027-02-28 23:59:59.999999")}

In [7]:
test_df = pd.DataFrame({
    'name': ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j'],
    'date1': [
        'Fall',
        'Spring',
        np.nan,
        'Fall',
        'Spring',
        np.nan,
        'Summer',
        'Spring',
        'Fall',
        'Summer',
    ],
    'date2': [
        'Winter',
        'Summer',
        np.nan,
        'Winter',
        'Summer',
        np.nan,
        'Fall',
        'Fall',
        'Winter',
        'Fall',
    ]
})

In [8]:
test_df

,name,date1,date2
0,a,Fall,Winter
1,b,Spring,Summer
2,c,NaN,NaN
3,d,Fall,Winter
4,e,Spring,Summer
5,f,NaN,NaN
6,g,Summer,Fall
7,h,Spring,Fall
8,i,Fall,Winter
9,j,Summer,Fall


In [9]:
test_df["range"] = test_df["date1"] + " to " + test_df["date2"]

In [10]:
test_df

,name,date1,date2,range
0,a,Fall,Winter,Fall to Winter
1,b,Spring,Summer,Spring to Summer
2,c,NaN,NaN,NaN
3,d,Fall,Winter,Fall to Winter
4,e,Spring,Summer,Spring to Summer
5,f,NaN,NaN,NaN
6,g,Summer,Fall,Summer to Fall
7,h,Spring,Fall,Spring to Fall
8,i,Fall,Winter,Fall to Winter
9,j,Summer,Fall,Summer to Fall


In [11]:
phenos["fruit range"] = phenos["Reproduction | Fruit/Seed Period Begin"] + " to " + phenos["Reproduction | Fruit/Seed Period End"]

In [12]:
phenos

,Unnamed: 0,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range,fruit range
0,0,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,No,Yellow,No,Green,Dense,Dense,Brown,No,Yes,Mid Summer,Medium,Fall,Fall,No,abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,Fall to Fall
1,1,Abies fraseri - fraser fir,abies fraseri,fraser fir,Abies,fraseri,ABFR,No,Purple,No,Dark Green,Moderate,Moderate,Brown,Yes,Yes,Mid Spring,Medium,Spring,Fall,No,abies fraseri,Non-flowering,N/a,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,Spring to Fall
2,2,Acer ginnala - amur maple,acer ginnala,amur maple,Acer,ginnala,ACGI,Yes,White,No,Green,Dense,Moderate,Brown,Yes,No,Mid Spring,High,Summer,Fall,No,acer ginnala,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-03-01', '2026-03-02', '20...",Summer to Fall
3,3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,Green,Dense,Porous,Brown,Yes,No,Early Spring,High,Summer,Fall,Yes,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-03-01', '2026-03-02', '20...",Summer to Fall
4,4,Acer nigrum - black maple,acer nigrum,black maple,Acer,nigrum,ACNI5,Yes,Yellow,No,Green,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,Yes,acer nigrum,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-04-01', '2026-04-02', '20...",Summer to Fall
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN
290,290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN
291,291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN,NaN
292,292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN


In [13]:
phenos.groupby("fruit range").size()

fruit range
Fall to Fall          8
Fall to Spring        2
Fall to Winter        1
Spring to Fall       10
Spring to Spring     18
Spring to Summer     20
Summer to Fall      101
Summer to Summer     24
Summer to Winter      2
Winter to Winter      1
dtype: int64

In [14]:
def split_season(date):
    if date != date:
        return []
    elif date == "Non-flowering":
        return []
    elif date == "N/a":
        return []
    else:
        pieces = re.split(r'\s+to\s+', date)
        normalized_pieces = [re.sub(r"\s+", "", piece) for piece in pieces]
        return normalized_pieces

In [15]:
def date_list(seasons: list):
    if len(seasons) > 1:
        rng = pd.date_range(start=season_start[seasons[0]],
                              end=season_end[seasons[1]], freq='1D')
        return rng
    elif len(seasons) == 1:
        rng = pd.date_range(start=season_start[seasons[0]],
                              end=season_end[seasons[0]], freq='1D')
        return rng
    else:
        return np.nan

In [16]:
phenos["fruit_range"] = phenos["fruit range"].apply(lambda d: date_list(split_season(d)))

In [17]:
phenos.loc[(phenos["Reproduction | Fruit/Seed Period End"].isna()) & (phenos["fruit_range"].isna())]

,Unnamed: 0,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range,fruit range,fruit_range
79,79,Ginkgo biloba - ginkgo,ginkgo biloba,ginkgo,Ginkgo,biloba,GIBI2,Yes,NaN,No,Yellow-Green,Porous,Porous,Yellow,No,No,NaN,NaN,NaN,NaN,No,ginkgo biloba,April,Green,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-04-01', '2026-04-02', '20...",NaN,NaN
80,80,Ginkgo biloba - princeton sentry ginkgo,ginkgo biloba,princeton sentry ginkgo,Ginkgo,biloba,GIBI2,Yes,NaN,No,Yellow-Green,Porous,Porous,Yellow,No,No,NaN,NaN,NaN,NaN,No,ginkgo biloba,April,Green,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-04-01', '2026-04-02', '20...",NaN,NaN
189,189,Acer palmatum - japanese maple,acer palmatum,japanese maple,Acer,palmatum,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,acer palmatum,April,Reddish-purple,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-04-01', '2026-04-02', '20...",NaN,NaN
190,190,Cornus kousa - kousa dogwood,cornus kousa,kousa dogwood,Cornus,kousa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cornus kousa,May to June,White to pinkish (bracts),Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-05-01', '2026-05-02', '20...",NaN,NaN
191,191,Amelanchier species - other serviceberry,amelanchier species,other serviceberry,Amelanchier,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,amelanchier species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NaN
290,290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NaN
291,291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN,NaN,NaN
292,292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NaN


In [18]:
phenos = phenos.drop(columns=["Unnamed: 0", "fruit range"])

In [21]:
phenos

,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,Morphology/Physiology | Foliage Porosity Summer,Morphology/Physiology | Foliage Porosity Winter,Morphology/Physiology | Fruit/Seed Color,Morphology/Physiology | Fruit/Seed Conspicuous,Morphology/Physiology | Leaf Retention,Reproduction | Bloom Period,Reproduction | Fruit/Seed Abundance,Reproduction | Fruit/Seed Period Begin,Reproduction | Fruit/Seed Period End,Suitability/Use | Palatable Human,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range,fruit_range
0,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,No,Yellow,No,Green,Dense,Dense,Brown,No,Yes,Mid Summer,Medium,Fall,Fall,No,abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"DatetimeIndex(['2026-09-01', '2026-09-02', '20..."
1,Abies fraseri - fraser fir,abies fraseri,fraser fir,Abies,fraseri,ABFR,No,Purple,No,Dark Green,Moderate,Moderate,Brown,Yes,Yes,Mid Spring,Medium,Spring,Fall,No,abies fraseri,Non-flowering,N/a,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"DatetimeIndex(['2026-03-01', '2026-03-02', '20..."
2,Acer ginnala - amur maple,acer ginnala,amur maple,Acer,ginnala,ACGI,Yes,White,No,Green,Dense,Moderate,Brown,Yes,No,Mid Spring,High,Summer,Fall,No,acer ginnala,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-03-01', '2026-03-02', '20...","DatetimeIndex(['2026-06-01', '2026-06-02', '20..."
3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,Green,Dense,Porous,Brown,Yes,No,Early Spring,High,Summer,Fall,Yes,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"DatetimeIndex(['2026-03-01', '2026-03-02', '20...","DatetimeIndex(['2026-06-01', '2026-06-02', '20..."
4,Acer nigrum - black maple,acer nigrum,black maple,Acer,nigrum,ACNI5,Yes,Yellow,No,Green,Dense,Porous,Brown,No,No,Late Spring,Medium,Summer,Fall,Yes,acer nigrum,N/a,N/a,N/a,N/a,no results container,NaN,"DatetimeIndex(['2026-04-01', '2026-04-02', '20...","DatetimeIndex(['2026-06-01', '2026-06-02', '20..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN
290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN
291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN,NaN
292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN


In [23]:
print(type(phenos.loc[0, "fruit_range"]))

<class 'pandas.DatetimeIndex'>


In [29]:
phenos.to_parquet("/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno_deaggregated.parquet")

ArrowKeyError: A type extension with name pandas.period already defined